# Pipeline météorologique robuste : médiane, leakage, calibration et export horaire
Ce notebook structure le pipeline demandé : fusion des sources ERA5 / Open-Meteo / NASA POWER avec données terrain, prétraitement robuste, features temporelles, détection de leakage, entraînement XGBoost, calibration et export CSV.

In [ ]:
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

DRIVE_ROOT = Path('C:/Users/Dell/Drive/MyDrive/MyDrive')
TERRAIN_ROOT = DRIVE_ROOT / 'Données Météo'
ERA5_PATH = DRIVE_ROOT / 'extraction_gee_20260423_154703.csv'
OPENMETEO_PATH = DRIVE_ROOT / 'extraction_open_meteo_20260423_160106.csv'
NASA_PATH = DRIVE_ROOT / 'extraction_nasa_power_20260423_164719.csv'
PROCESSED = DRIVE_ROOT / 'data_processed'
METADATA_EXCLUSIONS = [
    'datetime', 'date', 'time', 'year', 'month', 'day', 'hour', 'week', 'dayofweek',
    'latitude', 'longitude', 'lat', 'lon', 'location_id', 'station_id', 'region_id', 'region_name',
    'geometry', 'geom', 'shapefile', 'country', 'province',
 ]


## Chargement et fusion des données terrain, ERA5, Open-Meteo, NASA POWER
Ce bloc charge les sources brutes, normalise les colonnes, et prépare un DataFrame commun sur `region_id` et `datetime`.

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]
    return df

def load_terrain_data(root: Path) -> pd.DataFrame:
    files = list(root.glob('*.xls*'))
    frames = []
    for path in files:
        try:
            data = pd.read_excel(path, engine='openpyxl')
        except ValueError:
            data = pd.read_excel(path, engine='xlrd')
        data = normalize_columns(data)
        data['source'] = 'terrain'
        frames.append(data)
    return pd.concat(frames, ignore_index=True)

def load_satellite_data(paths: dict) -> pd.DataFrame:
    sources = []
    for source_name, path in paths.items():
        df = pd.read_csv(path)
        df = normalize_columns(df)
        if 'datetime' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime'])
        df['source'] = source_name
        sources.append(df)
    return pd.concat(sources, ignore_index=True, sort=False)

terrain = load_terrain_data(TERRAIN_ROOT)
satellite = load_satellite_data({'era5': ERA5_PATH, 'openmeteo': OPENMETEO_PATH, 'nasa': NASA_PATH})

terrain['datetime'] = pd.to_datetime(terrain.get('datetime', terrain.get('date')))
satellite['datetime'] = pd.to_datetime(satellite['datetime'])

combined = pd.concat([terrain, satellite], ignore_index=True, sort=False)
combined['region_id'] = combined.get('region_id', combined.get('location_id'))
combined['region_id'] = combined['region_id'].fillna('unknown')
combined = combined.sort_values('datetime').reset_index(drop=True)
print('Dimensions fusionnées :', combined.shape)


## Prétraitement robuste et conversion d’unités
Ce bloc convertit les unités courantes (Kelvin → Celsius, m → mm, J/m² → W/m²) et standardise les principales variables.

In [ ]:
def convert_units(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'temperature' in df.columns:
        df['temperature'] = df['temperature'].where(df['temperature'] < 200, df['temperature'] - 273.15)
    if 'precipitation' in df.columns:
        df['precipitation'] = df['precipitation'].where(df['precipitation'] < 10, df['precipitation'] * 1000)
    if 'radiation' in df.columns:
        df['radiation'] = df['radiation'].where(df['radiation'] < 2000, df['radiation'] / 3600)
    return df

combined = convert_units(combined)
combined = normalize_columns(combined)
combined.head(3)


## Imputation des valeurs manquantes avec la médiane locale par région/mois
On utilise une stratégie hiérarchique : interpolation linéaire region, ffill/bfill, médiane par region/mois, puis zéro en dernier recours.

In [ ]:
numeric_cols = combined.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['year', 'month', 'day', 'hour']]

def fill_missing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['month'] = df['datetime'].dt.month
    for col in numeric_cols:
        df[col] = df.groupby('region_id')[col].apply(lambda s: s.interpolate(method='linear', limit_direction='both'))
        df[col] = df.groupby('region_id')[col].ffill().bfill()
        region_median = df.groupby(['region_id', 'month'])[col].transform('median')
        month_median = df.groupby('month')[col].transform('median')
        df[col] = df[col].fillna(region_median).fillna(month_median).fillna(0)
    return df

combined = fill_missing(combined)
combined[numeric_cols].isna().sum().sum()


## Clipping des outliers et exclusion des métadonnées à risque de leakage
On applique le clipping IQR 1%-99% sur la température, et on retire les colonnes de metadata / identifiants du training.

In [ ]:
def clip_outliers(df: pd.DataFrame, col: str = 'temperature') -> pd.DataFrame:
    if col not in df.columns:
        return df
    q1, q99 = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(q1, q99)
    return df

def remove_metadata(df: pd.DataFrame, exclusions: list) -> pd.DataFrame:
    return df.drop(columns=[c for c in df.columns if c in exclusions], errors='ignore')

def drop_high_corr_features(df: pd.DataFrame, target: str, threshold: float = 0.97) -> list:
    numeric = df.select_dtypes(include='number')
    if target not in numeric.columns:
        return []
    corr = numeric.corr()[target].abs()
    return [col for col, val in corr.items() if val > threshold and col != target]

combined = clip_outliers(combined, 'temperature')
leakage_temp = drop_high_corr_features(combined, 'temperature')
leakage_precip = drop_high_corr_features(combined, 'precipitation')
print('Leakage temperature :', leakage_temp)
print('Leakage precipitation :', leakage_precip)
combined = remove_metadata(combined, METADATA_EXCLUSIONS)


## Feature engineering avec encodages temporels et lags
Création des encodages sin/cos, interactions saisonnières, consensus satellite, et mémoire thermique via lags.

In [ ]:
def build_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['month'] = df['datetime'].dt.month
    df['hour'] = df['datetime'].dt.hour
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
    df['hour_x_summer'] = df['hour'] * df['month'].isin([6, 7, 8]).astype(int)
    df['hour_x_winter'] = df['hour'] * df['month'].isin([12, 1, 2]).astype(int)
    return df

def build_lags(df: pd.DataFrame, col: str, lags: list) -> pd.DataFrame:
    df = df.sort_values(['region_id', 'datetime'])
    for lag in lags:
        df[f'{col}_t-{lag}'] = df.groupby('region_id')[col].shift(lag)
    return df

def build_consensus(df: pd.DataFrame) -> pd.DataFrame:
    sat_temp_cols = [c for c in df.columns if 'temp' in c and c != 'temperature']
    if sat_temp_cols:
        df['sat_temperature_mean'] = df[sat_temp_cols].mean(axis=1)
    return df

combined = build_time_features(combined)
combined = build_lags(combined, 'temperature', [1, 3, 6, 24])
combined = build_lags(combined, 'precipitation', [1, 6])
combined = build_consensus(combined)
combined = combined.dropna(subset=['temperature', 'precipitation'], how='any')
combined.head(3)


## Cibles 24h sur 8 pas de 3 heures
Construction des colonnes cibles pour les 8 pas suivants, puis suppression des lignes incomplètes.

In [ ]:
def build_24h_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['region_id', 'datetime'])
    for step in range(1, 9):
        df[f'temp_target_{step}'] = df.groupby('region_id')['temperature'].shift(-step * 3)
        df[f'precip_target_{step}'] = df.groupby('region_id')['precipitation'].shift(-step * 3)
    return df

combined = build_24h_targets(combined)
target_cols = [c for c in combined.columns if c.startswith('temp_target_') or c.startswith('precip_target_')]
combined = combined.dropna(subset=target_cols, how='any')
combined[target_cols].head(2)


## Split temporel et spatial + leakage automatique
Split chronologique 75/10/15, plus détection des colonnes de leakage corrélées à la cible.

In [ ]:
def temporal_split(df: pd.DataFrame, train_frac=0.75, val_frac=0.10):
    df = df.sort_values('datetime')
    n = len(df)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    return df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(), df.iloc[val_end:].copy()

train_df, val_df, test_df = temporal_split(combined)

def detect_leakage(df: pd.DataFrame, target: str, threshold=0.97):
    numeric = df.select_dtypes(include='number')
    if target not in numeric.columns:
        return []
    corr = numeric.corr()[target].abs()
    return [col for col, val in corr.items() if val > threshold and col != target]

leakage_temp = detect_leakage(train_df, 'temperature')
leakage_precip = detect_leakage(train_df, 'precipitation')
print('Leakage temperature :', leakage_temp)
print('Leakage precipitation :', leakage_precip)

plt.figure(figsize=(12, 4))
plt.plot(train_df['datetime'], train_df['temperature'], label='train', alpha=0.5)
plt.plot(val_df['datetime'], val_df['temperature'], label='val', alpha=0.7)
plt.plot(test_df['datetime'], test_df['temperature'], label='test', alpha=0.7)
plt.legend()
plt.title('Couverture temporelle : train / val / test')
plt.xlabel('datetime')
plt.ylabel('temperature')
plt.show()


## Entraînement XGBoost température et précipitation
Séparation des modèles, log1p pour précipitation, Optuna et early stopping pour limiter l’overfitting.

In [ ]:
feature_cols = [c for c in combined.columns if c not in target_cols + ['datetime', 'region_id', 'source']]
X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
y_train_temp = train_df['temperature']
y_val_temp = val_df['temperature']
y_train_precip = np.log1p(train_df['precipitation'])
y_val_precip = np.log1p(val_df['precipitation'])

def optimize_xgb(X, y, n_trials=60):
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 700),
            'max_depth': trial.suggest_int('max_depth', 3, 9),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'verbosity': 0,
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X, y)
        preds = model.predict(X)
        return mean_squared_error(y, preds, squared=False)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    best = study.best_params
    return xgb.XGBRegressor(**best, random_state=42, verbosity=0)

model_temp = optimize_xgb(X_train, y_train_temp)
model_temp.fit(X_train, y_train_temp, early_stopping_rounds=20, eval_set=[(X_val, y_val_temp)], verbose=False)

model_precip = optimize_xgb(X_train, y_train_precip)
model_precip.fit(X_train, y_train_precip, early_stopping_rounds=20, eval_set=[(X_val, y_val_precip)], verbose=False)

pred_temp = model_temp.predict(X_val)
pred_precip = np.expm1(model_precip.predict(X_val))

print('Temp RMSE:', mean_squared_error(y_val_temp, pred_temp, squared=False))
print('Temp R2:', r2_score(y_val_temp, pred_temp))
print('Precip RMSE:', mean_squared_error(np.expm1(y_val_precip), pred_precip, squared=False))
print('Precip R2:', r2_score(np.expm1(y_val_precip), pred_precip))


## Calibration et intervalles de confiance bootstrap
Correction biais par région/mois et calcul d’intervalles 95% sur bootstrap de 30 échantillons.

In [ ]:
def bias_correct(y_true, y_pred, groups):
    error = y_true - y_pred
    bias = error.groupby(groups).transform('median')
    return y_pred + bias

pred_temp_bias = bias_correct(val_df['temperature'], pred_temp, val_df['region_id'])
pred_precip_bias = bias_correct(np.expm1(val_df['precipitation']), pred_precip, val_df['region_id'])

def bootstrap_ic(model, X, transform=lambda x: x, n=30):
    preds = np.stack([transform(model.predict(X.sample(frac=1, replace=True))) for _ in range(n)], axis=1)
    lower = np.percentile(preds, 2.5, axis=1)
    upper = np.percentile(preds, 97.5, axis=1)
    return lower, upper

temp_lower, temp_upper = bootstrap_ic(model_temp, X_val, transform=lambda x: x)
precip_lower, precip_upper = bootstrap_ic(model_precip, X_val, transform=lambda x: np.expm1(x))

print('IC 95% calculés pour validation')


## Exportation des prédictions horaires au format CSV
Fonction pour générer un CSV de prédictions entre une date de début et une date de fin, avec intervalles de confiance.

In [ ]:
def export_hourly_predictions(model_temp, model_precip, df, feature_cols, output_path: Path):
    X = df[feature_cols]
    results = df[['datetime', 'region_id']].copy()
    results['temperature_pred'] = model_temp.predict(X)
    results['precipitation_pred'] = np.expm1(model_precip.predict(X))
    t_lower, t_upper = bootstrap_ic(model_temp, X, transform=lambda x: x)
    p_lower, p_upper = bootstrap_ic(model_precip, X, transform=lambda x: np.expm1(x))
    results['temperature_lower'] = t_lower
    results['temperature_upper'] = t_upper
    results['precipitation_lower'] = p_lower
    results['precipitation_upper'] = p_upper
    output_path.parent.mkdir(parents=True, exist_ok=True)
    results.to_csv(output_path, index=False)
    return results

output_file = PROCESSED / 'predictions_hourly.csv'
exported = export_hourly_predictions(model_temp, model_precip, test_df, feature_cols, output_file)
print('Export CSV :', output_file)


## Visualisation du leakage et de la couverture d’entraînement
Affichage des lignes de leakage détectées et diagramme de la couverture train/validation/test.

In [ ]:
leakage_columns = list(set(leakage_temp + leakage_precip))
leakage_preview = combined.loc[:, combined.columns.intersection(leakage_columns)]
print('Aperçu des colonnes de leakage détectées :')
display(leakage_preview.head())

plt.figure(figsize=(12, 5))
plt.plot(train_df['datetime'], train_df['temperature'], label='train', alpha=0.5)
plt.plot(val_df['datetime'], val_df['temperature'], label='val', alpha=0.7)
plt.plot(test_df['datetime'], test_df['temperature'], label='test', alpha=0.7)
plt.legend()
plt.title('Couverture temporelle du jeu de données d’apprentissage')
plt.xlabel('datetime')
plt.ylabel('temperature')
plt.show()
